# Phone (HAR): reading the person, not the task

*Notebook journey for §4 of the dissertation. Every number the chapter uses is produced here, and each step prints what it did, so nothing is taken on trust. We follow the deep-learning basics faithfully first, then look hard at the data.*

## a. Reading the problem

In [1]:
import os, collections
import numpy as np

# the phone data (UCI HAR): each row is one short window of motion,
# already summarised for us as 561 numbers, with a label for the activity.
DATA = "data/har.npz" if os.path.exists("data/har.npz") else "../data/har.npz"
d = np.load(DATA, allow_pickle=True)
X, y, subj = d["X"], d["y"], d["subj"]
ACT = ["walking", "upstairs", "downstairs", "sitting", "standing", "laying"]

print("windows :", len(y))
print("features:", X.shape[1])
print("people  :", len(set(subj.tolist())), "(ids 1..30)")
print("classes :", len(ACT), "->", ACT)

# read ONE window from top to bottom, so we see exactly what a single row is
i = 0
print(f"\none window (row {i}):")
print("  belongs to person :", int(subj[i]))
print("  activity label    :", int(y[i]), "->", ACT[int(y[i])])
print("  561 features, first 6:", np.round(X[i, :6], 4).tolist())

# the structure: rows arrive in long runs of ONE person doing ONE thing
print("\nfirst 6 rows (person, activity):")
for i in range(6):
    print(f"  row {i}: person {int(subj[i])}, {ACT[int(y[i])]}")

# how much data per person, and the do-nothing baseline
c = collections.Counter(subj.tolist())
print("\nwindows per person: min", min(c.values()), "max", max(c.values()),
      "mean", round(sum(c.values()) / len(c), 1))
cc = collections.Counter(y.tolist())
top, n = cc.most_common(1)[0]
print("commonest activity:", ACT[top], "=> always-guess baseline", round(n / len(y), 3))
print("uniform chance    :", round(1 / len(ACT), 3))

windows : 10299
features: 561
people  : 30 (ids 1..30)
classes : 6 -> ['walking', 'upstairs', 'downstairs', 'sitting', 'standing', 'laying']

one window (row 0):
  belongs to person : 1
  activity label    : 4 -> standing
  561 features, first 6: [0.2886, -0.0203, -0.1329, -0.9953, -0.9831, -0.9135]

first 6 rows (person, activity):
  row 0: person 1, standing
  row 1: person 1, standing
  row 2: person 1, standing
  row 3: person 1, standing
  row 4: person 1, standing
  row 5: person 1, standing

windows per person: min 281 max 409 mean 343.3
commonest activity: laying => always-guess baseline 0.189
uniform chance    : 0.167


## b. A first run: does it even learn?

We follow the book: scale the 561 features, build the small network from Section 1, and train. First with the learning rate that worked on the market, then with a smaller one. We watch the training accuracy, epoch by epoch, to see whether it settles or falls apart.

In [2]:
# the model: the same small network from Section 1 (one hidden layer, ReLU, softmax)
def init(d, width, K, rng):
    return [rng.standard_normal((d, width)) * 0.1, np.zeros(width),
            rng.standard_normal((width, K)) * 0.1, np.zeros(K)]

def forward(X, p):
    W1, b1, W2, b2 = p
    a = X @ W1 + b1
    h = np.maximum(a, 0.0)                  # ReLU
    z = h @ W2 + b2
    return a, h, z

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def accuracy(p, X, y):
    return float((forward(X, p)[2].argmax(1) == y).mean())

def train(Xtr, ytr, width, lr, epochs, K, rng, report=None):
    p = init(Xtr.shape[1], width, K, rng)
    n = len(ytr)
    Y = np.zeros((n, K)); Y[np.arange(n), ytr] = 1.0
    for e in range(epochs):
        a, h, z = forward(Xtr, p)
        dz = (softmax(z) - Y) / n           # the gap, p - y  (derived in Appendix E)
        W1, b1, W2, b2 = p
        dW2 = h.T @ dz;  db2 = dz.sum(0)
        da  = (dz @ W2.T) * (a > 0)          # back through ReLU
        dW1 = Xtr.T @ da; db1 = da.sum(0)
        p = [W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2]
        if report and (e % report == 0 or e == epochs - 1):
            print(f"    epoch {e:>3}: train accuracy {accuracy(p, Xtr, ytr):.3f}")
    return p

# scale the 561 features, then follow the book and train
Xs = (X - X.mean(0)) / (X.std(0) + 1e-9)
K = 6

with np.errstate(over="ignore", invalid="ignore"):   # a diverging run overflows; hush the noise
    print("step too big (learning rate 0.5, the value that worked on the market):")
    train(Xs, y, width=64, lr=0.5, epochs=150, K=K, rng=np.random.default_rng(1), report=30)
    print("\nstep made smaller (learning rate 0.1):")
    train(Xs, y, width=64, lr=0.1, epochs=150, K=K, rng=np.random.default_rng(1), report=30)

step too big (learning rate 0.5, the value that worked on the market):
    epoch   0: train accuracy 0.441


    epoch  30: train accuracy 0.002


    epoch  60: train accuracy 0.337


    epoch  90: train accuracy 0.167


    epoch 120: train accuracy 0.000


...

    epoch  30: train accuracy 0.878


    epoch  60: train accuracy 0.931


    epoch  90: train accuracy 0.948


    epoch 120: train accuracy 0.957


    epoch 149: train accuracy 0.962


## c. The number that was too good

Two ways to split the same data: shuffle the rows (the book's way), or hold out whole people (the way the model would really be used, on someone new). Same network, same training. We put them side by side.

In [3]:
def standardize(Xtr, Xte):                 # fit on the training side only
    m, s = Xtr.mean(0), Xtr.std(0) + 1e-9
    return (Xtr - m) / s, (Xte - m) / s

def split_records(seed):                   # by the book: shuffle rows, hold out a random fifth
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y)); nte = len(y) // 5
    return idx[nte:], idx[:nte]

def split_subjects(seed):                  # hold out whole PEOPLE: 6 of the 30 go entirely to test
    rng = np.random.default_rng(seed)
    test_subs = rng.permutation(np.unique(subj))[:6]
    te = np.isin(subj, test_subs)
    return np.where(~te)[0], np.where(te)[0]

def run_split(splitter, seed):
    tr, te = splitter(seed)
    Xtr, Xte = standardize(X[tr], X[te])
    p = train(Xtr, y[tr], width=64, lr=0.1, epochs=300, K=6, rng=np.random.default_rng(seed + 1))
    return accuracy(p, Xte, y[te])

rec = [run_split(split_records,  s) for s in range(5)]
sub = [run_split(split_subjects, s) for s in range(5)]
print("shuffle rows (record-wise) :", [round(a, 3) for a in rec], " mean", round(float(np.mean(rec)), 3))
print("hold out people (subject)  :", [round(a, 3) for a in sub], " mean", round(float(np.mean(sub)), 3))

shuffle rows (record-wise) : [0.962, 0.968, 0.968, 0.969, 0.964]  mean 0.966
hold out people (subject)  : [0.967, 0.963, 0.93, 0.946, 0.934]  mean 0.948


## d. Is it the activity, or the person?

The honest drop was small and shaky, so we need a cleaner test. Fix the exact same test windows, from the same handful of people, and change only one thing: whether the model was allowed to meet those people during training. Any difference now is identity, and nothing else.

In [4]:
# a fair test: fix the SAME test windows, change only whether the model met those people in training
def leak_control(seed):
    rng = np.random.default_rng(seed)
    test_people = rng.permutation(np.unique(subj))[:6]
    in_tp = np.isin(subj, test_people)
    tp = rng.permutation(np.where(in_tp)[0])
    nte = len(tp) // 5
    te    = tp[:nte]                       # the SAME test windows in both versions
    swing = tp[nte:]                       # the test people's OTHER windows
    others = np.where(~in_tp)[0]           # everyone else, always in training
    def acc(tr):
        Xtr, Xte = standardize(X[tr], X[te])
        p = train(Xtr, y[tr], width=64, lr=0.1, epochs=300, K=6, rng=np.random.default_rng(seed + 1))
        return accuracy(p, Xte, y[te])
    seen   = acc(np.concatenate([others, swing]))   # LEAKY: their other windows are in train
    unseen = acc(others)                            # HONEST: these people never appear in train
    return seen, unseen

pairs  = [leak_control(s) for s in range(5)]
seen   = [a for a, b in pairs]
unseen = [b for a, b in pairs]
gaps   = [a - b for a, b in pairs]
print("same test people, SEEN in training :", [round(a, 3) for a in seen],   " mean", round(float(np.mean(seen)), 3))
print("same test people, UNSEEN           :", [round(b, 3) for b in unseen], " mean", round(float(np.mean(unseen)), 3))
print("gap (identity leak, isolated)      :", [round(g, 3) for g in gaps],   " mean", round(float(np.mean(gaps)), 3))

same test people, SEEN in training : [0.977, 0.972, 0.972, 0.959, 0.963]  mean 0.968
same test people, UNSEEN           : [0.97, 0.953, 0.925, 0.952, 0.934]  mean 0.947
gap (identity leak, isolated)      : [0.008, 0.018, 0.047, 0.007, 0.029]  mean 0.022
